In [8]:
from dotenv import load_dotenv
from google import genai

load_dotenv()           # citeste cheia din fisierul .env
client = genai.Client() # gaseste singur cheia GEMINI_API_KEY

raspuns = client.models.generate_content(
    model="gemini-3.5-flash",
    contents="Explica in doua propozitii ce e o firma de logistica."
)
print(raspuns.text)

O firmă de logistică este o companie specializată în planificarea, transportul, depozitarea și distribuția mărfurilor de la producător până la consumatorul final. 

Rolul ei principal este să se asigure că produsele ajung la destinație în siguranță și la timpul potrivit, optimizând costurile și resursele întregului lanț de aprovizionare.


In [9]:
import pandas as pd, ast
df = pd.read_json("companies.jsonl", lines=True)

def verifica(cerere, descriere):
    prompt = f"""
    Esti un analist care verifica daca o firma se potriveste cu cererea unui utilizator.

Cererea utilizatorului: {cerere}

Descrierea firmei: {descriere}

Reguli:
- O firma e DA doar daca face activitatea ceruta ca afacere principala, pentru clienti. Nu ajunge sa mentioneze domeniul in descriere.
- Firmele care vand echipamente sau servicii PENTRU domeniul cerut (furnizori), sau care cumpara de la el (clienti), sunt NU.
- Firmele care fac activitatea doar pentru propriile produse (de exemplu isi transporta singure marfa) sunt NU.
- Verifica toate conditiile din cerere (tara, domeniu etc.). Daca descrierea contrazice o conditie, raspunde NU.

Raspunde exact asa:
Primul rand: doar DA sau NU
Al doilea rand: motivul, intr-o propozitie
    """
    raspuns = client.models.generate_content(model="gemini-3.5-flash", contents=prompt)
    return raspuns.text

In [10]:
cerere = "Logistic companies in Romania"
print(verifica(cerere, df.description[21]))  # STILL
print(verifica(cerere, df.description[10]))  # Portul Constanta

NU
Firma este un furnizor de echipamente și soluții tehnologice pentru intralogistică, nu o companie care oferă servicii propriu-zise de logistică și transport pentru clienți.
DA
Compania are sediul în România și oferă ca activitate principală servicii de manipulare a mărfurilor, operațiuni portuare și servicii de tranzit pentru clienți, încadrându-se direct în domeniul logisticei.


In [11]:
for i in [2, 1, 13, 19]:   # Rompetrol, Unilever, METRO, Valbur
    print(df.operational_name[i])
    print(verifica(cerere, df.description[i]))
    print("---")

Rompetrol
NU
Rompetrol este o companie din domeniul energetic ce își gestionează rețeaua de logistică doar pentru sprijinirea propriilor operațiuni, nu ca serviciu principal oferit altor clienți.
---
Unilever
NU
Compania este specializată în distribuția și comercializarea propriilor bunuri de consum, nu în furnizarea de servicii de logistică pentru clienți terți.
---
METRO România
NU
Compania este un comerciant cu ridicata care își gestionează logistica doar pentru propriile produse și operațiuni, neavând ca afacere principală furnizarea de servicii de logistică pentru clienți.
---
Valbur
NU
Firma este specializată în facilitarea comerțului și a investițiilor (intermediere), având ca activitate principală tranzacționarea și consultanța, nu furnizarea directă de servicii de logistică.
---


observam ca la cazurile la limita, chiar și eticheta umana e discutabilă

In [12]:
def to_dict(x):
    if isinstance(x, str):
        return ast.literal_eval(x)
    return x

df["address"] = df.address.apply(to_dict)
df["country"] = df.address.str.get("country_code")
ro = df[df.country == "ro"].copy()
len(ro)

26

In [13]:
eticheta_da = [6, 10, 12, 15, 19, 20]   # CFR, Port, Romgaz, Posta, Valbur, Brasov
ro["eticheta"] = ro.index.isin(eticheta_da)

este gresit acest raspuns, deoarece CFR furnizeaza servicii pentru clienti, transportul acestora, sau a bunurilor acestora

modelul a gresit pentru că i a lipsit informatia, nu pentru ca a rationat prost

In [14]:
def verifica_lot(cerere, firme):
    # firme = un tabel cu firmele de verificat
    lista = ""
    for i in firme.index:
        lista += f"[{i}] {firme.operational_name[i]}: {firme.description[i]} Oferte: {firme.core_offerings[i]}\n\n"

    prompt = f"""
Esti un analist care verifica daca fiecare firma din lista se potriveste cu cererea unui utilizator.

Cererea utilizatorului: {cerere}

Reguli:
- O firma e DA doar daca face activitatea ceruta ca afacere principala, pentru clienti. Nu ajunge sa mentioneze domeniul in descriere.
- Firmele care vand echipamente sau servicii PENTRU domeniul cerut (furnizori), sau care cumpara de la el (clienti), sunt NU.
- Firmele care fac activitatea doar pentru propriile produse (de exemplu isi transporta singure marfa) sunt NU.
- Verifica toate conditiile din cerere (tara, domeniu etc.). Daca descrierea contrazice o conditie, raspunde NU.

Firmele:
{lista}

Raspunde cu exact un rand pentru fiecare firma, in formatul:
[numar] DA sau NU | motivul intr-o propozitie
Nu scrie nimic altceva.
"""
    raspuns = client.models.generate_content(model="gemini-3.5-flash", contents=prompt)
    return raspuns.text

In [15]:
def citeste_raspuns(text):
    rezultate = {}
    for rand in text.strip().split("\n"):
        if rand.startswith("["):
            numar = int(rand[1:rand.index("]")])
            decizie = rand[rand.index("]")+1:].strip()
            rezultate[numar] = decizie
    return rezultate

In [16]:
import json, os

def verifica_cu_cache(cerere, firme, fisier):
    if os.path.exists(fisier):
        with open(fisier, encoding="utf-8") as f:
            return json.load(f)
    text = verifica_lot(cerere, firme)
    rezultate = citeste_raspuns(text)
    with open(fisier, "w", encoding="utf-8") as f:
        json.dump(rezultate, f, ensure_ascii=False, indent=2)
    return rezultate

In [17]:
rez = verifica_cu_cache(cerere, ro, "cache_cererea1.json")
len(rez)

26

In [18]:
ro["raspuns_llm"] = [rez.get(str(i), rez.get(i, "")) for i in ro.index]
ro["llm_da"] = ro.raspuns_llm.str.strip().str.startswith("DA")

In [1]:
%run solution.py

  answered by: gemini-3.5-flash
Query 1: Logistic companies in Romania
  candidates after pre-filter: 26 | qualified: 1
    [15] Poșta Română
  answered by: gemini-3.5-flash
Query 2: Public software companies with more than 1,000 employees
  candidates after pre-filter: 20 | qualified: 17
    [267] Tata Consultancy Services
    [269] Capgemini
    [270] Wipro
    [271] DWS Group
    [272] NTT DATA
    [274] Fujitsu
    [276] CGI
    [277] Atos
    [278] EPAM
    [279] Leidos
  answered by: gemini-3.5-flash
Query 3: Food and beverage manufacturers in France
  candidates after pre-filter: 20 | qualified: 20
    [380] La Péro
    [381] L'Essinganaise
    [382] Paniers d'Eve
    [383] LA MULTILOCALE
    [384] Yumi
    [385] Domaine Godeau
    [386] Famille Pouille
    [387] MILHADE Vins & Spiritueux
    [388] La Limonaderie de Paris
    [389] Auvergnat Cola
  answered by: gemini-3.5-flash
Query 4: Companies that could supply packaging materials for a direct-to-consumer cosmetics brand
  ca

In [3]:
%run evaluate.py

Query 1: found 1 | missed 0 | wrong 0 | precision 100% | recall 100%
Query 6: found 27 | missed 1 | wrong 1 | precision 96% | recall 96%
   missed: [450]
   wrong:  [471]
Query 12: found 42 | missed 1 | wrong 0 | precision 100% | recall 98%
   missed: [376]
